# Workshop 04 — Dice Rolling and Movement

Time to make the game interactive! In this workshop you will implement dice rolling, player movement around the board, and the "passed GO" bonus. By the end, players will be able to take turns rolling dice and moving to new spaces.

**What you will learn:**
- Connecting button clicks to game functions
- Modulo arithmetic for wrapping around the board
- Enabling and disabling buttons based on game phase
- Updating the log and display after each action

**Time:** roughly 15 to 20 minutes.

## Section 1 — Rolling the Dice

The `rollDice` function generates two random numbers between 1 and 6, then moves the current player. The `%` (modulo) operator makes the position wrap around when it goes past space 39 back to 0.

In [ ]:
import os

project_folder = os.path.join(os.path.expanduser("~"), "text_monopoly")
os.makedirs(project_folder, exist_ok=True)

html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Text Monopoly - Dice and Movement</title>
    <style>
        :root {
            --bg: #0d1117; --surface: #161b22; --border: #30363d;
            --text: #e6edf3; --dim: #8b949e; --green: #3fb950;
            --blue: #58a6ff; --red: #f85149; --yellow: #d29922; --purple: #bc8cff;
        }
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { background: var(--bg); color: var(--text); font-family: Georgia, serif; padding: 20px; }
        h1 { color: var(--green); margin-bottom: 4px; }
        .subtitle { color: var(--dim); font-size: 0.85rem; margin-bottom: 20px; }
        .game-container { display: grid; grid-template-columns: 1fr 320px; gap: 20px; max-width: 1100px; }
        .console { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 16px; height: 450px; overflow-y: auto; font-size: 0.82rem; line-height: 1.7; }
        .sidebar { display: flex; flex-direction: column; gap: 12px; }
        .card { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 16px; }
        .card h3 { margin-bottom: 8px; }
        .stat { display: flex; justify-content: space-between; padding: 3px 0; font-size: 0.85rem; }
        .label { color: var(--dim); }
        .dice-display { text-align: center; font-size: 2.5rem; padding: 8px; letter-spacing: 12px; }
        .turn-indicator { text-align: center; font-size: 0.85rem; padding: 8px; border-radius: 8px; font-weight: bold; }
        .turn-p1 { background: rgba(63, 185, 80, 0.15); color: var(--green); }
        .turn-p2 { background: rgba(88, 166, 255, 0.15); color: var(--blue); }
        button { width: 100%; padding: 12px; border: 1px solid var(--border); border-radius: 8px; background: var(--bg); color: var(--text); font-size: 0.9rem; cursor: pointer; margin-bottom: 8px; font-weight: bold; }
        button:hover:not(:disabled) { border-color: var(--green); }
        button:disabled { opacity: 0.3; cursor: not-allowed; }
        .roll-btn { background: var(--green); border: none; color: #fff; font-size: 1rem; }
        .roll-btn:hover:not(:disabled) { background: #2ea043; }
        .log-system { color: var(--dim); }
        .log-player1 { color: var(--green); }
        .log-player2 { color: var(--blue); }
        .log-event { color: var(--yellow); }
        .log-reward { color: var(--purple); }
    </style>
</head>
<body>
    <h1>Text Monopoly</h1>
    <p class="subtitle">Workshop 04 - dice rolling and movement</p>

    <div class="game-container">
        <div class="console" id="console"></div>
        <div class="sidebar">
            <div class="turn-indicator" id="turnIndicator">Player 1's Turn</div>
            <div class="dice-display" id="diceDisplay">🎲 🎲</div>
            <div class="card" id="p1card"></div>
            <div class="card" id="p2card"></div>
            <div class="card">
                <button class="roll-btn" id="rollBtn" onclick="rollDice()">🎲 Roll Dice</button>
                <button id="endTurnBtn" onclick="endTurn()" disabled>End Turn</button>
            </div>
        </div>
    </div>

    <script>
        // ---- BOARD DATA (abbreviated for readability) ----
        const BOARD = [
            { name: "GO",              type: "go" },
            { name: "Old Kent Road",   type: "property", cost: 60,  rent: 10 },
            { name: "Community Chest", type: "chest" },
            { name: "Whitechapel Rd",  type: "property", cost: 60,  rent: 10 },
            { name: "Income Tax",      type: "tax", amount: 200 },
            { name: "Kings Cross",     type: "property", cost: 200, rent: 50 },
            { name: "Angel Islington", type: "property", cost: 100, rent: 20 },
            { name: "Chance",          type: "chance" },
            { name: "Euston Road",     type: "property", cost: 100, rent: 20 },
            { name: "Pentonville Rd",  type: "property", cost: 120, rent: 25 },
            { name: "Just Visiting",   type: "visiting" },
            { name: "Pall Mall",       type: "property", cost: 140, rent: 30 },
            { name: "Electric Co.",    type: "property", cost: 150, rent: 35 },
            { name: "Whitehall",       type: "property", cost: 140, rent: 30 },
            { name: "Northumberland",  type: "property", cost: 160, rent: 35 },
            { name: "Marylebone Stn",  type: "property", cost: 200, rent: 50 },
            { name: "Bow Street",      type: "property", cost: 180, rent: 40 },
            { name: "Community Chest", type: "chest" },
            { name: "Marlborough St",  type: "property", cost: 180, rent: 40 },
            { name: "Vine Street",     type: "property", cost: 200, rent: 45 },
            { name: "Free Parking",    type: "free" },
            { name: "Strand",          type: "property", cost: 220, rent: 50 },
            { name: "Chance",          type: "chance" },
            { name: "Fleet Street",    type: "property", cost: 220, rent: 50 },
            { name: "Trafalgar Sq",    type: "property", cost: 240, rent: 55 },
            { name: "Fenchurch Stn",   type: "property", cost: 200, rent: 50 },
            { name: "Leicester Sq",    type: "property", cost: 260, rent: 60 },
            { name: "Coventry Street", type: "property", cost: 260, rent: 60 },
            { name: "Water Works",     type: "property", cost: 150, rent: 35 },
            { name: "Piccadilly",      type: "property", cost: 280, rent: 65 },
            { name: "GO TO JAIL",      type: "gotojail" },
            { name: "Regent Street",   type: "property", cost: 300, rent: 70 },
            { name: "Oxford Street",   type: "property", cost: 300, rent: 70 },
            { name: "Community Chest", type: "chest" },
            { name: "Bond Street",     type: "property", cost: 320, rent: 75 },
            { name: "Liverpool Stn",   type: "property", cost: 200, rent: 50 },
            { name: "Chance",          type: "chance" },
            { name: "Park Lane",       type: "property", cost: 350, rent: 85 },
            { name: "Super Tax",       type: "tax", amount: 100 },
            { name: "Mayfair",         type: "property", cost: 400, rent: 100 }
        ];

        // ---- GAME STATE ----
        let state = {
            players: [
                { name: "Player 1", cash: 1500, position: 0, properties: [], inJail: false, jailTurns: 0 },
                { name: "Player 2", cash: 1500, position: 0, properties: [], inJail: false, jailTurns: 0 }
            ],
            currentPlayer: 0,
            dice: [0, 0],
            phase: "roll",
            gameOver: false,
            turnCount: 0
        };

        // ---- LOG FUNCTION ----
        function log(text, className) {
            const el = document.getElementById("console");
            const line = document.createElement("div");
            line.className = "log-" + (className || "system");
            line.textContent = "> " + text;
            el.appendChild(line);
            el.scrollTop = el.scrollHeight;
        }

        // ---- ROLL DICE ----
        function rollDice() {
            if (state.phase !== "roll" || state.gameOver) return;

            const d1 = Math.floor(Math.random() * 6) + 1;
            const d2 = Math.floor(Math.random() * 6) + 1;
            state.dice = [d1, d2];
            const total = d1 + d2;

            // Show dice emoji
            const diceEmoji = ['', '⚀', '⚁', '⚂', '⚃', '⚄', '⚅'];
            document.getElementById("diceDisplay").textContent = diceEmoji[d1] + " " + diceEmoji[d2];

            const p = state.players[state.currentPlayer];
            const pClass = state.currentPlayer === 0 ? "player1" : "player2";

            log(p.name + " rolled " + d1 + " + " + d2 + " = " + total, pClass);

            // Move the player
            const oldPos = p.position;
            p.position = (p.position + total) % 40;  // modulo wraps around

            // Did they pass GO?
            if (p.position < oldPos) {
                p.cash += 200;
                log("Passed GO! Collect £200", "reward");
            }

            const space = BOARD[p.position];
            log("Landed on: " + space.name, pClass);

            state.phase = "endturn";
            updateUI();
        }

        // ---- END TURN ----
        function endTurn() {
            if (state.phase !== "endturn") return;
            state.currentPlayer = (state.currentPlayer + 1) % 2;
            state.phase = "roll";
            state.turnCount++;
            updateUI();
        }

        // ---- UPDATE UI ----
        function updateUI() {
            // Turn indicator
            const p = state.currentPlayer;
            const ti = document.getElementById("turnIndicator");
            ti.textContent = state.players[p].name + "'s Turn";
            ti.className = "turn-indicator turn-p" + (p + 1);

            // Player cards
            const colours = ["var(--green)", "var(--blue)"];
            for (let i = 0; i < 2; i++) {
                const pl = state.players[i];
                document.getElementById("p" + (i+1) + "card").innerHTML =
                    "<h3 style='color:" + colours[i] + "'>" + pl.name + "</h3>" +
                    "<div class='stat'><span class='label'>Cash</span><span>£" + pl.cash + "</span></div>" +
                    "<div class='stat'><span class='label'>Position</span><span>" + BOARD[pl.position].name + "</span></div>" +
                    "<div class='stat'><span class='label'>In Jail</span><span>" + (pl.inJail ? "Yes" : "No") + "</span></div>";
            }

            // Button states
            document.getElementById("rollBtn").disabled = state.phase !== "roll";
            document.getElementById("endTurnBtn").disabled = state.phase !== "endturn";
        }

        // ---- INIT ----
        log("Welcome to Text Monopoly! 🎩");
        log("Each player starts with £1500.");
        log("Roll the dice to begin!");
        updateUI();
    </script>
</body>
</html>
"""

filepath = os.path.join(project_folder, "04_dice_and_movement.html")
with open(filepath, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"File saved: {filepath}")
print("Open it and play! Roll dice, then end turn, repeat.")

## Section 2 — How Modulo Works

The line `p.position = (p.position + total) % 40` is important. The `%` operator gives the *remainder* after division. So if a player is on space 38 and rolls a 5:

- 38 + 5 = 43
- 43 % 40 = 3

They end up on space 3, having wrapped around past GO. This is how circular boards work in code.

## Section 3 — How the Phase System Works

The `state.phase` variable controls which buttons are active:

| Phase | Roll button | End Turn button | What happens |
|---|---|---|---|
| `"roll"` | Enabled | Disabled | Player rolls dice |
| `"endturn"` | Disabled | Enabled | Player ends their turn |

Later we will add a `"buy"` phase when a player lands on an unowned property. This simple state machine prevents players from clicking buttons at the wrong time.

## Section 4 — Detecting "Passed GO"

We check if the player passed GO by comparing the new position with the old one. If the new position is *less than* the old position, it means the player wrapped around the board. When this happens, they collect £200.

```javascript
if (p.position < oldPos) {
    p.cash += 200;
}
```

This is a simple but effective approach that works because the board is a one-way loop.

## Section 5 — Challenge

1. Add dice emoji to the log message (hint: use the `diceEmoji` array that is already defined).
2. Make the "Passed GO" check also work when the player lands *exactly* on GO (currently it only triggers when passing over it).
3. Add a turn counter display in the sidebar that shows how many turns have been played.
4. Try adding a console.log statement inside `rollDice` and open your browser's Developer Tools (F12) to see it. This is how developers debug!

In **Workshop 05** you will add property buying and rent collection. The game is taking shape! 🎯